In [1]:
from langchain_aws import ChatBedrockConverse, BedrockEmbeddings
import os
from typing import TypedDict
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.runnables import RunnableLambda
from langchain_core.prompts import ChatPromptTemplate


/home/labuser/.local/lib/python3.10/site-packages/langgraph/cache/base/__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [2]:
#Configure AWS Bedrock credentials
import os
#os.environ["AWS_ACCESS_KEY_ID"]=" "
#os.environ["AWS_SECRET_ACCESS_KEY"]= " "
# ---------------------------
# AWS Bedrock LLM Setup
# ---------------------------
llm = ChatBedrockConverse(model_id="amazon.nova-lite-v1:0", region_name="us-east-1", temperature=0.5, max_tokens=50)


In [3]:
# 1. Define State
class AgentState(TypedDict):
    """The state of the graph. We'll use a simple history of messages."""
    messages: list[HumanMessage | AIMessage] # Use the core types
    question: str
## 2. Define Hooks (Simple functions that run logic)
def pre_model_hook(state: AgentState) -> AgentState:
    """Executed BEFORE the LLM is invoked."""
    print("--- Pre-Model Hook started ---")
    print(f"Input question: {state.get('question', 'N/A')}")
    return state # Always return the (potentially modified) state
def post_model_hook(state: AgentState) -> AgentState:
    """Executed AFTER the LLM is invoked."""
    print("---  Post-Model Hook stared ---")
    # Check the last message in the history for the LLM's response
    if state.get("messages") and state["messages"][-1].content:
        print(f"LLM Response snippet: '{state['messages'][-1].content[:50]}...'")
    print("---------------------------------")
    return state


In [4]:
# 3. Define the LLM Node Logic
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Be concise and conversational."),
    ("user", "{question}")
])
llm_runnable = prompt_template | llm
# This function updates the state by invoking the LLM.
def llm_executor(state: AgentState) -> AgentState:
    """Invokes the LLM and updates the state with the result."""
    question = state["question"]
    ai_message = llm_runnable.invoke({"question": question})
    new_messages = state.get("messages", []) + [ai_message]
    # The output from the node is the updated state dictionary
    return {"messages": new_messages, "question": question}


In [5]:
## 4. Hook the components together in a sequence
# We define a new function that represents the sequence of (Pre-Hook -> LLM Execution -> Post-Hook)
def llm_node_with_hooks(state: AgentState) -> AgentState:
    """Combines the hooks and the LLM execution logic sequentially."""
    # 1. Run Pre-Hook
    state = pre_model_hook(state)
    # 2. Run LLM Executor (main logic)
    state = llm_executor(state)
    # 3. Run Post-Hook
    state = post_model_hook(state)
    return state
# 5. Define the Initial Input Node
def initial_input(state: AgentState) -> AgentState:
    """Sets the initial question and message history."""
    initial_q = "What is the capital of India and how does LangGraph work?"
    return {
        "question": initial_q,
        "messages": [HumanMessage(content=initial_q)]
    }


In [6]:
# 6. Build and Compile the Graph
graph = StateGraph(AgentState)
graph.add_node("setup_input", initial_input)
# The new, combined function is added as a single node
graph.add_node("llm_and_hooks", RunnableLambda(llm_node_with_hooks))
graph.set_entry_point("setup_input")
graph.add_edge("setup_input", "llm_and_hooks")
graph.add_edge("llm_and_hooks", END)
agent_graph = graph.compile()
# Run the graph
final_state = agent_graph.invoke({})
# Print Final Output
print("\n=== Final Agent Output ===")
if final_state.get("messages"):
    print(final_state["messages"][-1].content)
else:
    print("No final message was generated.")


--- Pre-Model Hook started ---
Input question: What is the capital of India and how does LangGraph work?
---  Post-Model Hook stared ---
LLM Response snippet: 'The capital of India is New Delhi. 

LangGraph, in...'
---------------------------------

=== Final Agent Output ===
The capital of India is New Delhi. 

LangGraph, in the context of programming and AI, is a framework for building complex language models. It allows developers to create graphs of language processing tasks, making it easier to manage and execute sequences of operations
